# Attention

Attention ist nicht nur ein Teil des Transformers. Attention IST
der Transformer.

Betrachte diesen Satz: Die Katze saß auf der Matte, weil es warm
war. Worauf bezieht sich „es“? Ein Mensch weiß, dass die Matte
gemeint ist. Aber wie würde ein Computer das herausfinden?

Bevor es Attention gab, las ein Modell Wörter eines nach dem
anderen von links nach rechts. Bis es beim Wort „es“ ankam, lag
das Wort „Matte“ weit in der Vergangenheit. Seine Information war
verblasst. Das Modell hätte es vergessen.

Attention löst dieses Problem, indem jedes Wort gleichzeitig auf
jedes andere Wort blicken kann. Wenn das Modell beim Wort „es“
ankommt, kann es auf alle vorherigen Wörter zurückblicken und
entscheiden, welches davon am wichtigsten ist. In diesem Fall
würde es auf „Matte“ schauen, weil Wärme eine Eigenschaft von
Objekten ist, nicht von Tieren.

Wie Attention entscheidet, wie viel Gewicht jedes Wort bekommt,
geschieht über sogenannte Queries, Keys und Values. Man kann es
sich wie auf einer Party vorstellen. Deine Query ist das, wonach
du suchst. Jede andere Person hat einen Key, der beschreibt, was
sie zu bieten hat. Eine hohe Übereinstimmung bedeutet, dass du
dieser Person Aufmerksamkeit schenkst.

In diesem Notebook bauen wir den vollständigen Attention-
Mechanismus von Grund auf. Wir verfolgen ein durchgerechnetes
Beispiel mit echten Zahlen, damit du genau siehst, was in jedem
modernen Sprachmodell im Inneren passiert.

## Der Maßstab dieses Notebooks

Wir bauen Attention in derselben Größe, wie sie GPT-2 Small
verwendet hat. Das bedeutet ein d_model von 768 und num_heads von
12. Jeder Head bekommt 64 Dimensionen zum Arbeiten. Der
vollständige Attention-Layer hat etwa 2,3 Millionen Parameter.

Das ist klein genug, um auf einem Laptop zu laufen, aber groß
genug, um dir zu zeigen, wie echte Sprachmodelle funktionieren.
Die Mathematik ändert sich bei größeren Dimensionen nicht. Nur die
Zahlen werden größer.

Hier ist, was sich beim Hochskalieren ändert. Die Architektur
bleibt gleich. Nur die Dimensionen wachsen. Diese Zahlen stammen
aus dem originalen GPT-3-Paper, das von OpenAI veröffentlicht
wurde.

| Modell | Layer | d_model | Heads | Parameter |
|---|---|---|---|---|
| GPT-2 Small | 12 | 768 | 12 | 124M |
| GPT-2 Medium | 24 | 1024 | 16 | 350M |
| GPT-2 Large | 36 | 1280 | 20 | 774M |
| GPT-3 350M | 24 | 1024 | 16 | 350M |
| GPT-3 1.3B | 24 | 2048 | 24 | 1.3B |
| GPT-3 6.7B | 32 | 4096 | 32 | 6.7B |
| GPT-3 175B | 96 | 12288 | 96 | 175B |
| LLaMA 7B | 32 | 4096 | 32 | 7B |
| LLaMA 13B | 40 | 5120 | 40 | 13B |
| LLaMA 70B | 80 | 8192 | 64 | 70B |

GPT-3 175B verwendet exakt dieselbe Attention-Formel, die wir
gleich schreiben werden. Es wendet sie nur mit 96 Layern statt 12
und mit 12288 Dimensionen statt 768 an. Der Code ist identisch.
Die Zahlen sind nur tausendmal größer.

Du kannst GPT-3 175B nicht auf einer Consumer-GPU ausführen. Es
benötigt mehrere Rechenzentrums-GPUs und Millionen von Dollar an
Stromkosten. Aber das Notebook, das du gleich ausführst, erfasst
dieselbe Idee in einer Größe, mit der du tatsächlich
experimentieren und verstehen kannst.

## Importe

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## Rotary Position Embeddings

Wir benötigen RoPE aus dem vorherigen Kapitel. Es rotiert Query-
und Key-Vektoren, sodass der Attention-Score von der relativen
Position abhängt.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even"
        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = freqs.repeat_interleave(2, dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    @staticmethod
    def rotate_half(x):
        x_even = x[..., 0::2]
        x_odd  = x[..., 1::2]
        return torch.stack([-x_odd, x_even], dim=-1).flatten(-2)
    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)

## Causal Mask

Während des Trainings kann jeder Token nur die Token sehen, die
vor ihm liegen. Das verhindert, dass das Modell schummelt, indem
es auf zukünftige Wörter schaut.

In [ ]:
def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

## Multi-Head Attention

Dies ist das vollständige Attention-Modul. Es erledigt acht
Schritte der Reihe nach. Projektion auf Q, K und V. Umformen in
mehrere Heads. RoPE anwenden. Skalierte Skalarprodukt-Scores
berechnen. Die Causal Mask anwenden. Softmax ausführen. Die
Values gewichten. Heads zusammenführen und die Ausgabe
projizieren.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = self.rotary(q, seq_len)
        k = self.rotary(k, seq_len)

        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        attn_output = attn_weights @ v

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)

        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)
        return output

## Das Attention-Modul testen

Erstelle ein kleines Modell mit 12 Attention Heads und füttere es
mit einem Batch aus 8 Sequenzen von je 64 Token.

In [ ]:
attn = MultiHeadAttention(d_model=768, num_heads=12)

batch_size = 8
seq_len = 64
x = torch.randn(batch_size, seq_len, 768)
mask = create_causal_mask(seq_len, x.device)

output = attn(x, mask)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Input and output shapes match: {x.shape == output.shape}")
print()

params = sum(p.numel() for p in attn.parameters())
print(f"Attention parameters: {params:,}")

## Durchgerechnetes Beispiel mit echten Zahlen

Lass uns nun Attention anhand eines winzigen Satzes mit 3 Token
Schritt für Schritt nachvollziehen. Wir verwenden ein d_model von
4 und einen einzelnen Head, damit wir jede Zahl sehen können.

In [ ]:
tiny_attn = MultiHeadAttention(d_model=4, num_heads=1)

tokens = torch.tensor([[
    [0.5,  0.2, -0.3,  0.8],
    [0.1, -0.5,  0.7, -0.2],
    [0.9,  0.3, -0.1, -0.5],
]])
print(f"Input tokens shape: {tokens.shape}")
print()

with torch.no_grad():
    qkv = tiny_attn.qkv_proj(tokens)
    qkv = qkv.reshape(1, 3, 3, 1, 4)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]

print("Query vectors:")
for i in range(3):
    vals = [f"{v:.3f}" for v in q[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")
print()

print("Key vectors:")
for i in range(3):
    vals = [f"{v:.3f}" for v in k[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")
print()

scores = (q @ k.transpose(-2, -1)) / math.sqrt(4)
print("Attention scores (before mask and softmax):")
print(f"  Token 0: {[f'{v:.3f}' for v in scores[0, 0, 0].tolist()]}")
print(f"  Token 1: {[f'{v:.3f}' for v in scores[0, 0, 1].tolist()]}")
print(f"  Token 2: {[f'{v:.3f}' for v in scores[0, 0, 2].tolist()]}")

## Causal Mask und Softmax anwenden

Die Causal Mask setzt zukünftige Positionen auf negativ
unendlich. Nach dem Softmax werden diese Positionen zu null.
Jeder Token kann nur sich selbst und die Token beachten, die
vorher kamen.

In [ ]:
mask = create_causal_mask(3, scores.device)
scores_masked = scores.masked_fill(mask == 0, float('-inf'))
weights = F.softmax(scores_masked, dim=-1)

print("Attention weights (after mask and softmax):")
for i in range(3):
    row = [f"{v:.3f}" for v in weights[0, 0, i].tolist()]
    print(f"  Token {i} attends to: [{', '.join(row)}]")
print()
print("Token 0 sees only itself. Token 1 sees 0 and 1. Token 2 sees all three.")
print()

output = weights @ v
print("Output vectors (weighted sum of values):")
for i in range(3):
    vals = [f"{v:.3f}" for v in output[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")

## Die Causal-Attention-Matrix visualisieren

Jede Zeile ist ein Token. Jede Spalte zeigt, worauf er achtet.
Das obere Dreieck ist leer, weil zukünftige Token blockiert sind.

In [ ]:
print("Causal attention matrix (Token i attending to Token j):")
print("          T0      T1      T2")
for i in range(3):
    row = "  ".join([f"{weights[0, 0, i, j]:.3f}" for j in range(3)])
    print(f"T{i}       {row}")
print()
print("The upper right is zero. The model cannot see the future.")